# Gridlock 2.0 — Traffic Demand Prediction
**Flipkart Hackathon** | Bengaluru CCTV Traffic Data

**Objective:** Predict normalised traffic demand (0–1) for geohash-level cells across 15-minute time slots.

**Pipeline:**
1. EDA (target distribution, temporal patterns, spatial analysis)
2. Preprocessing (vectorised imputation, ordinal encoding)
3. Feature Engineering (6 key improvements)
4. Feature Selection (sqrt target for importance, variance + correlation filters)
5. Modelling (LightGBM + XGBoost + CatBoost ensemble, GroupKFold)
6. Residual Analysis (by RoadType and time_bucket)
7. Submission

In [ ]:
# ── Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_squared_error

from preprocessor      import preprocess
from feature_engineer  import build_features
from feature_selection import select_features
from train             import transform_target, invert_target, run_training

pd.set_option('display.max_columns', 60)
SEED = 42
BASE = '../dataset'

---
## 1. Exploratory Data Analysis

In [ ]:
# Load raw data for EDA
raw_train  = pd.read_csv(f'{BASE}/train.csv')
raw_test   = pd.read_csv(f'{BASE}/test.csv')
sample_sub = pd.read_csv(f'{BASE}/sample_submission.csv')

print('Train shape:', raw_train.shape)
print('Test  shape:', raw_test.shape)
raw_train.head()

In [ ]:
# Data types and missing values
print('=== Train dtypes & nulls ===')
print(pd.concat([
    raw_train.dtypes.rename('dtype'),
    raw_train.isnull().sum().rename('nulls'),
    (raw_train.isnull().mean()*100).round(2).rename('null%')
], axis=1))

In [ ]:
# Target distribution — raw, log1p, and sqrt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

d = raw_train['demand']
axes[0].hist(d, bins=60, color='steelblue', edgecolor='white')
axes[0].set_title(f'Demand (skew={d.skew():.2f})')
axes[0].set_xlabel('demand')

dl = np.log1p(d)
axes[1].hist(dl, bins=60, color='coral', edgecolor='white')
axes[1].set_title(f'log1p(demand) (skew={dl.skew():.2f})')
axes[1].set_xlabel('log1p(demand)')

ds = np.sqrt(d)
axes[2].hist(ds, bins=60, color='mediumseagreen', edgecolor='white')
axes[2].set_title(f'sqrt(demand) (skew={ds.skew():.2f})')
axes[2].set_xlabel('sqrt(demand)')

plt.suptitle('Target Distribution Comparison', y=1.02)
plt.tight_layout()
plt.show()
print(d.describe())

In [ ]:
# Cross-day correlation: day 48 vs day 49 (if available in train)
raw_train['hour']      = raw_train['timestamp'].str.split(':').str[0].astype(int)
raw_train['minute']    = raw_train['timestamp'].str.split(':').str[1].astype(int)
raw_train['time_slot'] = (raw_train['hour'] * 60 + raw_train['minute']) // 15

days = sorted(raw_train['day'].unique())
print(f'Days in training: {days[:5]} ... {days[-5:]}')
print(f'Number of unique days: {len(days)}')

if len(days) >= 2:
    d_last  = raw_train[raw_train['day'] == days[-1]][['geohash','time_slot','demand']]
    d_prev  = raw_train[raw_train['day'] == days[-2]][['geohash','time_slot','demand']]
    cross   = d_last.merge(d_prev, on=['geohash','time_slot'],
                            suffixes=('_last','_prev'))
    corr = cross['demand_last'].corr(cross['demand_prev'])
    print(f'\nCorrelation (day {days[-1]} vs day {days[-2]}) on (geohash, time_slot): {corr:.4f}')

In [ ]:
# Demand by hour of day
hourly = raw_train.groupby('hour')['demand'].mean()
plt.figure(figsize=(12, 4))
hourly.plot(kind='bar', color='steelblue')
plt.title('Average Demand by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Mean Demand')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Demand by categorical features
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['RoadType', 'Weather', 'LargeVehicles']):
    raw_train.groupby(col)['demand'].mean().sort_values().plot(
        kind='barh', ax=ax, color='steelblue'
    )
    ax.set_title(f'Demand by {col}')
    ax.set_xlabel('Mean Demand')
plt.tight_layout()
plt.show()

In [ ]:
# Geohash overlap: how many test geohashes are unseen in train?
train_hashes = set(raw_train['geohash'].unique())
test_hashes  = set(raw_test['geohash'].unique())
unseen       = test_hashes - train_hashes
print(f'Train geohashes: {len(train_hashes)}')
print(f'Test  geohashes: {len(test_hashes)}')
print(f'Unseen in test : {len(unseen)}  ({len(unseen)/len(test_hashes)*100:.1f}%)')
print(f'Unseen hashes  : {unseen}')

---
## 2. Preprocessing

In [ ]:
train, test, sample_sub = preprocess(
    f'{BASE}/train.csv',
    f'{BASE}/test.csv',
    f'{BASE}/sample_submission.csv'
)
print('Post-preprocessing nulls (train):')
print(train.isnull().sum()[train.isnull().sum() > 0])

---
## 3. Feature Engineering

**New features in this version:**
- `demand_same_slot_prev_day` — cross-day same-slot lag (day N-1 → day N, corr ~0.79)
- `demand_lag1/2/4` — real test lag lookup from (geohash, time_slot-1) in training data
- `highway_x_afternoon` — interaction for highest-RMSE error segment
- `geo_slot_mean_demand` — geohash × time_slot joint aggregation
- `demand_cv` — coefficient of variation per geohash
- KNN fallback for ~10 unseen test geohashes

In [ ]:
train, test = build_features(train, test)
print(f'Train: {train.shape}   Test: {test.shape}')
print('\nNew feature columns:')
new_cols = ['demand_same_slot_prev_day','highway_x_afternoon',
            'geo_slot_mean_demand','demand_cv','demand_lag1']
print(train[[c for c in new_cols if c in train.columns]].describe())

---
## 4. Feature Selection

In [ ]:
# Pre-compute sqrt target for importance scoring
train['demand_sqrt'] = transform_target(train['demand'].values)
print(f'sqrt(demand) skewness: {train["demand_sqrt"].skew():.4f}  '
      f'(raw demand skewness: {train["demand"].skew():.4f})')

selected_features, importance_df, X_train, X_test = select_features(
    train, test,
    corr_threshold=0.95,
    var_threshold=0.001,
)
y_train = train['demand'].values
print(f'Features selected: {len(selected_features)}')

In [ ]:
# Feature importance plot
plt.figure(figsize=(10, 9))
importance_df.head(25).sort_values('importance').plot(
    kind='barh', x='feature', y='importance',
    legend=False, color='steelblue', figsize=(10, 9)
)
plt.title('Top 25 Feature Importances (LightGBM on sqrt target)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

---
## 5. Modelling

**Key changes:**
- **GroupKFold(groups=geohash)** — prevents same-geohash leakage between folds
- **sqrt(demand) target** — reduces skewness 3.73 → 1.58, better model calibration
- **Sample weights** — highway×afternoon rows weighted 2× in LightGBM
- **Ensemble** — LightGBM + XGBoost + CatBoost, inverse-RMSE weighted
- **Inversion** — `pred_demand = np.clip(pred_sqrt ** 2, 0, 1)`

In [ ]:
pred_ensemble, oof_ensemble = run_training(
    train, test, X_train, X_test, output_dir='.'
)

---
## 6. Residual Analysis

In [ ]:
# OOF residual histogram
residuals = y_train - oof_ensemble
plt.figure(figsize=(10, 4))
plt.hist(residuals, bins=80, color='steelblue', edgecolor='white')
plt.axvline(0, color='red', linestyle='--')
plt.title('OOF Residuals (Ensemble)')
plt.xlabel('Residual (demand)')
plt.tight_layout()
plt.show()

In [ ]:
# RMSE by RoadType segment
seg_df = train[['RoadType','time_bucket','highway_x_afternoon']].copy()
seg_df['demand']      = y_train
seg_df['oof']         = oof_ensemble
seg_df['sq_err']      = (seg_df['demand'] - seg_df['oof'])**2

road_labels = {0:'Residential', 1:'Street', 2:'Highway'}
rmse_road = seg_df.groupby('RoadType')['sq_err'].mean().apply(np.sqrt)
rmse_road.index = rmse_road.index.map(road_labels)
rmse_road.sort_values().plot(kind='barh', color='coral', figsize=(8,3))
plt.title('OOF RMSE by RoadType')
plt.xlabel('RMSE')
plt.tight_layout()
plt.show()

In [ ]:
# RMSE by time_bucket segment
bucket_labels = {0:'Night',1:'Morn-Rush',2:'Mid-Morn',
                 3:'Lunch',4:'Afternoon',5:'Eve-Rush',6:'Night2'}
rmse_bucket = seg_df.groupby('time_bucket')['sq_err'].mean().apply(np.sqrt)
rmse_bucket.index = rmse_bucket.index.map(bucket_labels)
rmse_bucket.sort_values().plot(kind='barh', color='mediumseagreen', figsize=(8,4))
plt.title('OOF RMSE by Time Bucket')
plt.xlabel('RMSE')
plt.tight_layout()
plt.show()

In [ ]:
# Highway x Afternoon segment RMSE
if 'highway_x_afternoon' in seg_df.columns:
    rmse_ha = seg_df.groupby('highway_x_afternoon')['sq_err'].mean().apply(np.sqrt)
    rmse_ha.index = rmse_ha.index.map({0:'Other', 1:'Highway×Afternoon'})
    print('RMSE by highway_x_afternoon segment:')
    print(rmse_ha.round(6))

---
## 7. Generate Submission

In [ ]:
submission = sample_sub.copy()
submission['demand'] = pred_ensemble

# Sanity checks
assert submission['demand'].isnull().sum() == 0, 'NaN in submission!'
assert (submission['demand'] >= 0).all(), 'Negative demand!'
assert (submission['demand'] <= 1).all(), 'Demand > 1!'

submission.to_csv('submission.csv', index=False)
print('Submission saved → submission.csv')
print(submission['demand'].describe().round(6))
submission.head(10)

In [ ]:
# Prediction distribution sanity check
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_train,          bins=60, color='steelblue', edgecolor='white', alpha=0.7, label='Train')
axes[0].hist(pred_ensemble,    bins=60, color='coral',     edgecolor='white', alpha=0.7, label='Test Pred')
axes[0].set_title('Demand Distribution: Train vs Predicted')
axes[0].set_xlabel('demand')
axes[0].legend()

axes[1].scatter(oof_ensemble[:5000], y_train[:5000],
                alpha=0.3, s=8, color='steelblue')
axes[1].plot([0,1],[0,1], 'r--', lw=1)
axes[1].set_title('OOF Predicted vs Actual (sample)')
axes[1].set_xlabel('Predicted demand')
axes[1].set_ylabel('Actual demand')

plt.tight_layout()
plt.show()

---
## Summary

| Change | Rationale | Expected Impact |
|--------|-----------|----------------|
| GroupKFold(geohash) | Prevents same-geohash in train+val | More honest OOF metrics |
| sqrt(demand) target | Skewness 3.73 → 1.58 | Better model calibration, lower RMSE |
| Cross-day slot lag | Corr ~0.79 with demand | Strong predictive signal |
| Real test lag lookup | Actual observed values vs mean proxy | Higher lag feature quality |
| KNN for unseen geohashes | ~10 unseen test hashes | Eliminates global-mean fallback |
| highway×afternoon interaction | Highest-RMSE segment | Targeted error correction |
| CatBoost in ensemble | Symmetric trees complement LGBM/XGB | Diversity benefit |
| Vectorised imputation | No slow apply(axis=1) | ~5-10× faster preprocessing |
| Unified prefix encoding | Train-fitted, no test leakage | Consistency |
| geo×slot joint agg | Captures slot-specific spatial pattern | Additional signal |